# PN23 — anti-pair fractal lift

## tl;dr

One stored representative per reversible wheel-residue pair reconstructed both child directions exactly through five recursive lifts. The untouched `p=17` rung reconstructed all **92,160** residues modulo **510,510** from **46,080** stored pair representatives, with zero mismatches. Independent validation passed **40/40** checks. This is a lossless `2:1` state compression and exact ARA ridge crosswalk to wheel/CRT symmetry, not a constant-cost next-prime locator.


## Context & Methods

The test starts from the modulo-14 anti-pairs `(1,13)`, `(3,11)` and `(5,9)`. It carries only the lower member of each pair. For each new prime gate `p`, it locates the one killed copy on the carried side, reflects that location to predict the opposite collision, and builds the next-rung pair representatives from the carried side alone.

Development gates are `3,5,11,13`; `17` is held out.

### Key Assumptions

- Residue reversal is `r ↔ M-r` for even wheel modulus `M`.
- A pair's ARA child-copy coordinate is normalized to `[0,2]`.
- Passing means exact equality with direct coprimality enumeration, not visual similarity.
- The sealed 87-bit prime anchor is outside this structural test.


In [1]:
import json
import math
from pathlib import Path

HERE = Path.cwd()
results = json.loads((HERE / 'PN23_ANTI_PAIR_FRACTAL_LIFT_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN23_ANTI_PAIR_FRACTAL_LIFT_VALIDATION.json').read_text(encoding='utf-8'))
print(results['status'])
print('independent validation:', validation['status'], validation['checks_passed'], '/', validation['checks_total'])
assert results['decision']['all_rungs_pass'] is True
assert validation['status'] == 'PASS'


PASS — LOSSLESS RECURSIVE ANTI-PAIR COMPRESSION
independent validation: PASS 40 / 40


## Data

The data are exact integer residue sets at each frozen wheel rung. No probabilistic prime sample or fitted parameter is used.


In [2]:
print('phase | lift | parent pairs | child pairs | residues | direct ridges | max error')
for row in results['rungs']:
    print(
        row['phase'],
        f"{row['parent_modulus']}x{row['gate']}->{row['new_modulus']}",
        row['parent_pair_count'],
        row['child_pair_count'],
        row['child_residue_count'],
        row['direct_ridge_count'],
        row['max_ridge_error'],
    )


phase | lift | parent pairs | child pairs | residues | direct ridges | max error
development 14x3->42 3 6 12 1 0.0
development 42x5->210 6 24 48 2 0.0
development 210x11->2310 24 240 480 3 0.0
development 2310x13->30030 240 2880 5760 18 0.0
held_out 30030x17->510510 2880 46080 92160 170 0.0


## Results

First verify exact rung growth, reconstruction and the fixed `2:1` lane compression.


In [3]:
for row in results['rungs']:
    assert row['pass'] is True
    assert row['child_pair_count'] == row['parent_pair_count'] * (row['gate'] - 1)
    assert row['child_residue_count'] == 2 * row['child_pair_count']
    assert row['child_residue_count'] == row['direct_residue_count']
    assert row['missing_residue_count'] == 0
    assert row['extra_residue_count'] == 0
    assert row['collision_failure_count'] == 0
    assert row['integer_ridge_failure_count'] == 0
    assert row['max_ridge_error'] == 0.0
    assert row['stored_lane_compression_ratio'] == 2.0
print('All frozen rung identities verified.')


All frozen rung identities verified.


In [4]:
smallest = [
    row for row in results['worked_paths']
    if row['parent_modulus'] == 14 and row['gate'] == 3
]
for row in smallest:
    print(
        f"A/B={row['parent_representative_A']}/{row['reconstructed_parent_B']}",
        f"k=({row['killed_copy_A']},{row['predicted_killed_copy_B']})",
        f"x=({row['x_A']:.1f},{row['x_B']:.1f})",
        f"mean={row['ridge_mean']:.1f}",
        'children=', row['next_pair_representatives_from_A_only'],
    )
    assert row['ridge_mean'] == 1.0


A/B=1/13 k=(1,1) x=(1.0,1.0) mean=1.0 children= [1, 13]
A/B=3/11 k=(0,2) x=(0.0,2.0) mean=1.0 children= [11, 17]
A/B=5/9 k=(2,0) x=(2.0,0.0) mean=1.0 children= [5, 19]


In [5]:
held_out = [row for row in results['rungs'] if row['phase'] == 'held_out'][0]
print('held-out gate:', held_out['gate'])
print('stored pair representatives:', held_out['child_pair_count'])
print('full residues reconstructed:', held_out['child_residue_count'])
print('missing / extra:', held_out['missing_residue_count'], held_out['extra_residue_count'])
assert held_out['gate'] == 17
assert held_out['child_pair_count'] == 46080
assert held_out['child_residue_count'] == 92160


held-out gate: 17
stored pair representatives: 46080
full residues reconstructed: 92160
missing / extra: 0 0


## Takeaways

1. One anti-pair adult representative is sufficient to reconstruct the opposite direction and all next-rung pair children exactly.
2. Child asymmetry can close to an exact adult `1.0` ridge: direct `(1,1)` and coarse `(0,2)/(2,0)` cases are distinct but share the same pair mean.
3. The compression is exactly `2:1`; it removes the redundant reflected half of the state.
4. The number of distinct child identities still grows by `p-1` at each new prime gate. Therefore PN23 validates the recursive ARA coordinate but not a two-number or constant-cost prime predictor.
